### 0) Imports and warning supression

In [1]:
from pathlib import Path
import sys
import numpy as np
import torch
import json
import warnings
import pandas as pd

repo_root = Path(r"C:/Git/aerosol_bnn")          # adjust
run_dir   = repo_root / "release" / "chosen_model"  # adjust (defaults to best as per hyperparm sweep)

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# import the new API
from release.inference_api import run_inference_latent, run_inference_phys
import release.config as cfg

warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", UserWarning)

C:\Users\dona965\AppData\Local\anaconda3\envs\bnn_notebook\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1) Load precompted taus for scaling

In [2]:
taus_path = run_dir / "taus.json"
taus = None
if taus_path.exists():
    with taus_path.open("r") as f:
        taus = json.load(f)
taus

{'Qext': 0.520647, 'SSA': 0.44035, 'g': 0.530355}

### 2) Run Inference

In [3]:
import torch as _torch
meta = _torch.load(run_dir / "data_meta.pt", map_location="cpu", weights_only=False)

Din = meta["x_test_raw"].shape[1]
print("Din =", Din)
print("Targets =", cfg.TARGETS)

N = 16
x = torch.rand(N, Din)  # dummy raw inputs

in_cols = list(cfg.FEATURES) + [f"{t}_HS" for t in cfg.TARGETS]

print("\nExpected input column order (x_raw):")
print("Din =", len(in_cols))
for i, c in enumerate(in_cols):
    print(f"  [{i:02d}] {c}")

Din = 8
Targets = ['Qext', 'SSA', 'g']

Expected input column order (x_raw):
Din = 8
  [00] Npp
  [01] V/V0
  [02] coating_RI_imag
  [03] Xve
  [04] core_Df
  [05] Qext_HS
  [06] SSA_HS
  [07] g_HS


### 2.1) Latent variable outputs

In [4]:
mu_lat, std_ale_lat, std_epi_lat, mu_lat_s, std_ale_lat_s = run_inference_latent(
    run_dir, x, num_mc=50, seed=0, taus=taus, return_samples=True
)

print("mu_lat:", mu_lat.shape)
print("std_ale_lat:", None if std_ale_lat is None else std_ale_lat.shape)
print("std_epi_lat:", std_epi_lat.shape)

mu_lat[:2]

mu_lat: torch.Size([16, 3])
std_ale_lat: torch.Size([16, 3])
std_epi_lat: torch.Size([16, 3])


tensor([[ 0.7530, -1.0593,  0.9858],
        [ 0.0130, -0.8549,  0.3750]])

### 2.2 Physical space outputs

In [5]:
mean_phys, std_ale_phys, std_epi_phys, std_tot_phys, q_out = run_inference_phys(
    run_dir, x, num_mc=50, seed=0, taus=taus, L=50, quantiles=(0.05, 0.5, 0.95)
)

print("mean_phys:", mean_phys.shape)
print("std_tot_phys:", std_tot_phys.shape)
print("q_out keys:", list(q_out.keys()))
print("Targets:", cfg.TARGETS)

mean_phys[:2], std_tot_phys[:2]

mean_phys: (16, 3)
std_tot_phys: (16, 3)
q_out keys: ['q05', 'q50', 'q95']
Targets: ['Qext', 'SSA', 'g']


(array([[2.2000334 , 0.26527095, 0.7209652 ],
        [1.0287802 , 0.30276224, 0.5903907 ]], dtype=float32),
 array([[0.6371754 , 0.07616146, 0.08978866],
        [0.18188979, 0.07308795, 0.07325722]], dtype=float32))

In [6]:
targets = cfg.TARGETS

# build a output table
out = {}
for j, t in enumerate(targets):
    out[f"mean_{t}"] = mean_phys[:, j]
    out[f"std_tot_{t}"] = std_tot_phys[:, j]
    out[f"std_ale_{t}"] = std_ale_phys[:, j]
    out[f"std_epi_{t}"] = std_epi_phys[:, j]
    out[f"q05_{t}"] = q_out["q05"][:, j]
    out[f"q50_{t}"] = q_out["q50"][:, j]
    out[f"q95_{t}"] = q_out["q95"][:, j]

df_results = pd.DataFrame(out)

# show a few rows
df_results.head().style.format("{:.6f}")

,mean_Qext,std_tot_Qext,std_ale_Qext,std_epi_Qext,q05_Qext,q50_Qext,q95_Qext,mean_SSA,std_tot_SSA,std_ale_SSA,std_epi_SSA,q05_SSA,q50_SSA,q95_SSA,mean_g,std_tot_g,std_ale_g,std_epi_g,q05_g,q50_g,q95_g
0,2.200033,0.637175,0.153624,0.618378,1.356590,2.060815,3.089379,0.265271,0.076161,0.026848,0.071272,0.148930,0.260082,0.387824,0.720965,0.089789,0.022467,0.086932,0.543778,0.735660,0.840481
1,1.028780,0.181890,0.069502,0.168087,0.773144,1.003944,1.361876,0.302762,0.073088,0.028815,0.067168,0.193797,0.296168,0.428925,0.590391,0.073257,0.026913,0.068134,0.464042,0.598996,0.693357
2,1.447501,0.471512,0.102237,0.460295,0.838354,1.367128,2.256468,0.021945,0.011076,0.003300,0.010573,0.007788,0.020544,0.043360,0.523501,0.128943,0.027005,0.126083,0.248508,0.535824,0.701071
3,0.999340,0.223779,0.070112,0.212512,0.714332,0.945826,1.448972,0.115777,0.038056,0.013989,0.035391,0.061925,0.111157,0.182439,0.531612,0.078856,0.027736,0.073818,0.377834,0.543026,0.639218
4,2.196313,0.716967,0.158080,0.699322,1.245696,1.991304,3.531336,0.051145,0.024030,0.007411,0.022859,0.018534,0.046840,0.094983,0.677926,0.118090,0.023871,0.115652,0.441422,0.697810,0.830608


### 2.2.1 Sanity check of error propogation

In [7]:
max_err = np.max(np.abs(std_tot_phys**2 - (std_ale_phys**2 + std_epi_phys**2)))
print(max_err)

7.1525574e-07
